In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from deepface import DeepFace
from imutils import paths
import tkinter as tk
import time
import pygame
import mediapipe as mp
import tensorflow as tf
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.layers import Dense,Layer,Flatten
from tensorflow.keras.utils import custom_object_scope
from concurrent.futures import ThreadPoolExecutor
import random
import string
import warnings
warnings.filterwarnings('ignore',category=DeprecationWarning)
labels=tf.constant([1,0,1])
logits=tf.constant([[2.0,1.0,0.1],[0.5,1.2,2.3],[3.1,0.2,1.5]])
loss=tf.keras.losses.sparse_categorical_crossentropy(labels,logits,from_logits=True)
class customLayer(Layer):
    def __init__(self,**kwargs):
        super(customLayer,self).__init__(**kwargs)
    def call(self,inputs):
        return inputs
model_path='D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\Models\\EfficientNetB0.keras'
model_save=Sequential([Flatten(input_shape=(224,224,3)),Dense(1,activation='sigmoid')])
model_save.save(model_path)
model1=cv2.CascadeClassifier("C:\\Python312\\Lib\\site-packages\\cv2\\data\\haarcascade_frontalface_default.xml")
model2=cv2.CascadeClassifier("C:\\Python312\\Lib\\site-packages\\cv2\\data\\haarcascade_fullbody.xml")
model3=cv2.CascadeClassifier("C:\\Python312\\Lib\\site-packages\\cv2\\data\\haarcascade_upperbody.xml")
model4=cv2.CascadeClassifier("C:\\Python312\\Lib\\site-packages\\cv2\\data\\haarcascade_lowerbody.xml")
sound_file="D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\buzzer.mp3"
excel_file="D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\PROJECT_DATA.xlsx"
c=0
c2=0
choice=0
duration=500
interval=0.5
name={}
t1=0
t2=0
data={"Id":[],"Name":[],"Age":[],"Gender":[],"Race":[]}
data2={"Name":[],"Age":[],"Gender":[],"Race":[],"Test":[]}
pygame.mixer.init()
buzzzer_sound=pygame.mixer.Sound(sound_file)
start_time=time.time()
last_blink_time=start_time
is_visible=True
""" def get_name(id):
    name=str(input(f"Enter name of Person {id} : "))
    return name """
def get_name(existing_name,length=8):
    while True:
        name=''.join(random.choices(string.ascii_letters+string.digits,k=length))
        if name not in existing_name:
            return name
def findage(videos):
    videos=cv2.cvtColor(videos,cv2.COLOR_BGR2RGB)
    result_age=DeepFace.analyze(videos,actions=["age"],enforce_detection=False)
    return str(result_age[0]['age'])
def findgen(videos):
    videos=cv2.cvtColor(videos,cv2.COLOR_BGR2RGB)
    result_gender=DeepFace.analyze(videos,actions=["gender"],enforce_detection=False)
    return str(result_gender[0]['dominant_gender'])
def findrace(videos):
    videos=cv2.cvtColor(videos,cv2.COLOR_BGR2RGB)
    result_race=DeepFace.analyze(videos,actions=["race"],enforce_detection=False)
    return str(result_race[0]['dominant_race'])
def findemo(videos):
    videos=cv2.cvtColor(videos,cv2.COLOR_BGR2RGB)
    result_emotion=DeepFace.analyze(videos,actions=["emotion"],enforce_detection=False)
    return str(result_emotion[0]['dominant_emotion'])
def preprocessed_image(videos):
    image=cv2.resize(videos,(224,224))
    image=np.expand_dims(image,axis=0)
    image=image/255.0
    return image
def predict_monkeypox(model,videos):
    result=model.predict(videos)
    return result
def scanning(model,videos):
    processed_image=preprocessed_image(videos)
    prediction =predict_monkeypox(model,processed_image)
    print("Prediction=",prediction[0][0])
    if prediction[0][0]>0.7:
        return 1
    else:
        return 0
def buzzer(x,y,current_time,last_blink_time,is_visible,message):
    if (current_time-start_time)<duration:
        if (current_time-last_blink_time)>interval:
            is_visible=not is_visible
            last_blink_time=current_time
            if is_visible and message=="!! Positive !!":
                buzzzer_sound.set_volume(1.0)
                buzzzer_sound.play()
        if is_visible:
            cv2.putText(videos,str(message),(x-80,y-80),cv2.FONT_HERSHEY_COMPLEX,1.5,(0,0,139),3)
    return last_blink_time,is_visible
def append_value(name,age,gender,race,message):
    data2["Name"].append(name)
    data2["Age"].append(age)
    data2["Gender"].append(gender)
    data2["Race"].append(race)
    if message=="!! Positive !!":
            data2["Test"].append("Positive")
    else:
            data2["Test"].append("Negative")
def append_value2(id,name,age,gender,race):
    data["Id"].append(id)
    data["Name"].append(name)
    data["Age"].append(age)
    data["Gender"].append(gender)
    data["Race"].append(race)
def excel_data():
    try:
        existing_data=pd.read_excel(excel_file)
    except FileNotFoundError:
        existing_data=pd.DataFrame()
    new_data=pd.DataFrame(data2)
    update_data=pd.concat([existing_data,new_data],ignore_index=True)
    if update_data.shape[1]>16384:
        update_data=update_data.iloc[:,:16384]
    update_data.to_excel(excel_file,index=False)
if model1.empty():
    print("Error!!")
camera=cv2.VideoCapture(0)
mp_holistic=mp.solutions.holistic
mp_drawing=mp.solutions.drawing_utils
with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)as holistic:
    while True:
        ret,videos=camera.read()
        if not ret:
            break
        rgb_col=cv2.cvtColor(videos,cv2.COLOR_BGR2RGB)
        col=cv2.cvtColor(videos,cv2.COLOR_BGR2GRAY)
        body=model2.detectMultiScale(
            col,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30,30),
            maxSize=(300,300),
            flags=cv2.CASCADE_SCALE_IMAGE
        )
        for (x,y,w,h) in body:
            t1=x
            t2=y
            cv2.rectangle(videos,(x,y),(x+w,y+h),(255,0,0),2)
        face=model1.detectMultiScale(
            col,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30,30),
            maxSize=(300,300),
            flags=cv2.CASCADE_SCALE_IMAGE
        )
        n=len(face)
        for (x,y,w,h) in face:
            t1=x
            t2=y
            face_roi=videos[y:y+h,x:x+w]
            if n>c and c2==0:
                c+=1
                c2=1
                existing_name=set(name.values())
                user_name=get_name(existing_name)
                name[c]=user_name
                message="!! Negative !!"
                text_size1=cv2.getTextSize(name[c],cv2.FONT_HERSHEY_SIMPLEX,0.5,2)[0]
                ag=findage(face_roi)
                text_size2=cv2.getTextSize(ag,cv2.FONT_HERSHEY_SIMPLEX,0.5,2)[0]
                gen=findgen(face_roi)
                text_size3=cv2.getTextSize(gen,cv2.FONT_HERSHEY_SIMPLEX,0.5,2)[0]
                ra=findrace(face_roi)
                text_size5=cv2.getTextSize(ra,cv2.FONT_HERSHEY_SIMPLEX,0.5,2)[0]
                append_value2(c,name,ag,gen,ra)
                emo=findemo(videos)
                text_size4=cv2.getTextSize(emo,cv2.FONT_HERSHEY_SIMPLEX,0.5,2)[0]
            #fh=int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
            cv2.rectangle(videos,(x,y),(x+w,y+h),(255,0,0),2)
            cv2.rectangle(videos,(x+w+15,y-(text_size1[1])-10),(x+w+15+(text_size1[0])+10,y),(0,36,255),-1)
            cv2.putText(videos,str(name[c]),(x+w+20,y-10),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),2)
            cv2.rectangle(videos,(x+w+15,y+(h//2)-(text_size2[1])-8),(x+w+15+(text_size2[0])+10,y+(h//2)),(0,36,255),-1)
            cv2.putText(videos,str(findage(face_roi)),(x+w+20,y+(h//2)),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),2)
            cv2.rectangle(videos,(x+w+15,y+h-(text_size3[1])-10),(x+w+15+(text_size3[0])+10,y+h),(0,36,255),-1)
            cv2.putText(videos,str(findgen(face_roi)),(x+w+20,y+h-10),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),2)
            cv2.rectangle(videos,(x-(text_size4[0])-25,y-(text_size4[1])-5),(x-10,y+5),(0,36,255),-1)
            cv2.putText(videos,str(findemo(face_roi)),(x-(text_size4[0])-20,y),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),2)
            cv2.rectangle(videos,(x-(text_size5[0])-25,y+(h//2)-(text_size5[1])-5),(x-10,y+(h//2)+5),(0,36,255),-1)
            cv2.putText(videos,str(findrace(face_roi)),(x-(text_size5[0])-20,y+(h//2)),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),2)
            current_time=time.time()
            last_blink_time,is_visible=buzzer(x,y,current_time,last_blink_time,is_visible,message)
        upper_body=model3.detectMultiScale(
            col,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30,30),
            maxSize=(300,300),
            flags=cv2.CASCADE_SCALE_IMAGE
        )
        for (x,y,w,h) in upper_body:
            t1=x
            t2=y
            cv2.rectangle(videos,(x,y),(x+w,y+h),(255,0,0),2)
        lower_body=model4.detectMultiScale(
            col,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30,30),
            maxSize=(300,300),
            flags=cv2.CASCADE_SCALE_IMAGE
        )
        for (x,y,w,h) in lower_body:
            t1=x
            t2=y
            cv2.rectangle(videos,(x,y),(x+w,y+h),(255,0,0),2)
        result=holistic.process(rgb_col)
        if result.pose_landmarks and result.face_landmarks and result.left_hand_landmarks and result.right_hand_landmarks:
            landmarks=result.pose_landmarks.landmark
            region={"chest":(landmarks[mp_holistic.PoseLandmark.LEFT_SHOULDER],landmarks[mp_holistic.PoseLandmark.RIGHT_SHOULDER]),
                    "stomach":(landmarks[mp_holistic.PoseLandmark.LEFT_HIP],landmarks[mp_holistic.PoseLandmark.RIGHT_HIP]),
                    "left_leg":(landmarks[mp_holistic.PoseLandmark.LEFT_HIP],landmarks[mp_holistic.PoseLandmark.LEFT_KNEE]),
                    "right_leg":(landmarks[mp_holistic.PoseLandmark.RIGHT_HIP],landmarks[mp_holistic.PoseLandmark.RIGHT_KNEE]),
                    "left_land":(result.left_hand_landmarks.landmark[mp_holistic.HandLandmark.WRIST]),
                    "right_hand":(result.right_hand_landmarks.landmark[mp_holistic.HandLandmark.WRIST])
                   }
            h,w,_=videos.shape
            for region_name,point in region.items():
                if isinstance(point,tuple) and len(point)>1:
                    x1,y1=int(point[0].x*w),int(point[0].y*h)
                    x2,y2=int(point[1].x*w),int(point[1].y*h)
                    x,y=(x1+x2)//2,(y1+y2)//2
                    t1=x
                    t2=y
                    cv2.rectangle(videos,(x-20,y-20),(x+20,y+20),(255,0,0),2)
                else:
                    continue
        if result.pose_landmarks:
            mp_drawing.draw_landmarks(
                videos,
                result.pose_landmarks,
                mp_holistic.POSE_CONNECTIONS,
            )
        if result.face_landmarks:
            mp_drawing.draw_landmarks(
                videos,
                result.face_landmarks,
                mp_holistic.FACEMESH_CONTOURS,
            )
        if result.left_hand_landmarks:
            mp_drawing.draw_landmarks(
                videos,
                result.left_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
            )
        if result.right_hand_landmarks:
            mp_drawing.draw_landmarks(
                videos,
                result.right_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
            )
        model=tf.keras.models.load_model(model_path)
        if model is None:
            print("ERROR")
        ch=scanning(model,videos)
        if ch==1:
            message="!! Positive !!"
        else:
            message="!! Negative !!"
        current_time=time.time()
        last_blink_time,is_visible=buzzer(t1,t2,current_time,last_blink_time,is_visible,message)
        if choice<c:
            append_value(name,ag,gen,ra,message)
            choice+=1
            c2=0
        cv2.imshow("Face_Detection",videos)
        if cv2.waitKey(1)==ord("a"):
            break
excel_data()
cv2.namedWindow("Face_Detection",cv2.WINDOW_NORMAL)
cv2.setWindowProperty("Face_Detection",cv2.WND_PROP_TOPMOST,1)
camera.release()
cv2.destroyAllWindows()


pygame 2.6.1 (SDL 2.28.4, Python 3.12.8)
Hello from the pygame community. https://www.pygame.org/contribute.html
1/1 [==============================] - 0s 385ms/step
Prediction= 0.49832946
1/1 [==============================] - 0s 338ms/step
Prediction= 0.6046324
1/1 [==============================] - 0s 74ms/step
Prediction= 0.5364763
1/1 [==============================] - 0s 59ms/step
Prediction= 0.5385503
1/1 [==============================] - 0s 64ms/step
Prediction= 0.479145
1/1 [==============================] - 0s 62ms/step
Prediction= 0.51608807
1/1 [==============================] - 0s 63ms/step
Prediction= 0.511209
1/1 [==============================] - 0s 60ms/step
Prediction= 0.56690544
1/1 [==============================] - 0s 64ms/step
Prediction= 0.5446257
1/1 [==============================] - 0s 71ms/step
Prediction= 0.5722476
1/1 [==============================] - 0s 96ms/step
Prediction= 0.5598865
1/1 [==============================] - 0s 60ms/step
Prediction= 0.558